# Практическая работа №3
## Автоматическая классификация учебных материалов образовательного портала с использованием трансформерной модели

**Цель:** обучить русскоязычный Transformer автоматически определять предмет учебного материала по его заголовку и тексту.

**Практическая ценность для ОбрПортал:** после обучения модель можно использовать для автоматического предложения категории при добавлении материала, а её encoder — как основу для будущего семантического поиска.

**Исследовательский вопрос:** насколько эффективно предварительно обученная трансформерная модель может классифицировать русскоязычные учебные материалы после fine-tuning на специализированном наборе данных?

### 1. План эксперимента

1. Загружаем открытый набор уроков **its5Q/resh-edu**.
2. Очищаем HTML и приводим названия предметов к единой системе классов.
3. Формируем сбалансированный датасет и делим его на train / validation / test.
4. Fine-tuning модели **cointegrated/rubert-tiny2** для sequence classification.
5. Оцениваем Accuracy, Precision, Recall, Macro-F1 и confusion matrix.
6. Анализируем ошибки.
7. Сохраняем модель и демонстрируем функцию, которую можно перенести в ML-сервис ОбрПортал.
8. Дополнительно показываем, как encoder модели может дать embeddings для семантического поиска.

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn seaborn beautifulsoup4 huggingface_hub sentencepiece

In [ ]:
import os, re, json, gzip, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from bs4 import BeautifulSoup
from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer,
    set_seed
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

### 2. Данные

Используем открытый русскоязычный набор уроков `its5Q/resh-edu`. В нём есть поля `subject`, `title`, `grade` и `summary`. Полный raw-файл достаточно большой, поэтому при чтении сохраняем только нужные поля и не держим упражнения в памяти.

In [ ]:
DATASET_REPO = "its5Q/resh-edu"
RAW_FILE = "raw.jsonl.gz"

raw_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename=RAW_FILE,
    repo_type="dataset"
)
print("Файл:", raw_path)
print("Размер, MB:", round(os.path.getsize(raw_path) / 1024**2, 1))

In [ ]:
records = []
with gzip.open(raw_path, "rt", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        records.append({
            "id": obj.get("id"),
            "subject": obj.get("subject"),
            "title": obj.get("title"),
            "grade": obj.get("grade"),
            "summary": obj.get("summary"),
        })

df_raw = pd.DataFrame(records)
print("Уроков:", len(df_raw))
display(df_raw.head(3))
print("\nТоп исходных предметов:")
display(df_raw["subject"].value_counts().head(25).to_frame("count"))

### 3. Предобработка и формирование классов

Для ОбрПортал полезнее не хранить десятки почти одинаковых вариантов названий предметов, а привести их к каноническим категориям. Например, алгебра и геометрия объединяются в **Математику**. Для модели используем заголовок урока и очищенный от HTML текст описания.

In [ ]:
def clean_html(value):
    if value is None:
        return ""
    text = BeautifulSoup(str(value), "html.parser").get_text(" ", strip=True)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def canonical_subject(value):
    s = str(value).strip().lower().replace("ё", "е")
    rules = [
        ("Русский язык", ["русск", "russian language"]),
        ("Информатика", ["информат", "computer science", "informatics"]),
        ("Математика", ["математ", "алгеб", "геометр", "math"]),
        ("Физика", ["физик", "physics"]),
        ("Химия", ["хими", "chemistry"]),
        ("Биология", ["биолог", "biology"]),
        ("История", ["истор", "history"]),
    ]
    for label, keys in rules:
        if any(k in s for k in keys):
            return label
    return None

df = df_raw.copy()
df["category"] = df["subject"].map(canonical_subject)
df["title_clean"] = df["title"].map(clean_html)
df["summary_clean"] = df["summary"].map(clean_html)
df["text"] = (df["title_clean"] + ". " + df["summary_clean"]).str.strip()
df = df[df["category"].notna() & (df["text"].str.len() >= 40)].copy()

counts = df["category"].value_counts()
print("Доступные целевые категории:")
display(counts.to_frame("count"))

In [ ]:
DESIRED = ["Математика", "Физика", "Информатика", "История", "Биология", "Химия", "Русский язык"]
MIN_CLASS_SIZE = 80
selected = [c for c in DESIRED if counts.get(c, 0) >= MIN_CLASS_SIZE]

if len(selected) < 5:
    raise ValueError(f"Найдено только {len(selected)} классов с >= {MIN_CLASS_SIZE} примерами: {selected}")

n_per_class = min(500, min(int(counts[c]) for c in selected))
parts = [df[df["category"] == c].sample(n=n_per_class, random_state=SEED) for c in selected]
df_model = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Классы:", selected)
print("Примеров на класс:", n_per_class)
print("Всего:", len(df_model))
display(df_model[["subject", "category", "grade", "title_clean"]].head(10))

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df_model, x="category", order=selected, color="#4C78A8")
plt.title("Распределение классов после балансировки")
plt.xlabel("Предмет"); plt.ylabel("Количество")
plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

lengths = df_model["text"].str.len()
print(lengths.describe(percentiles=[.5, .75, .9, .95]).round(1))

### 4. Train / validation / test

Разделение выполняется **стратифицированно**, поэтому доля каждого предмета одинакова во всех частях. Тестовая выборка не участвует ни в обучении, ни в выборе лучшего checkpoint.

In [ ]:
train_df, temp_df = train_test_split(
    df_model, test_size=0.30, random_state=SEED, stratify=df_model["category"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["category"]
)

labels = sorted(selected)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

for part in (train_df, val_df, test_df):
    part["labels"] = part["category"].map(label2id).astype(int)

print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
print("label2id =", label2id)

In [ ]:
def to_hf(frame):
    return Dataset.from_pandas(frame[["text", "labels"]].reset_index(drop=True), preserve_index=False)

datasets = DatasetDict({
    "train": to_hf(train_df),
    "validation": to_hf(val_df),
    "test": to_hf(test_df),
})
datasets

### 5. Transformer и токенизация

Используем `cointegrated/rubert-tiny2` — компактный русскоязычный BERT-encoder. Для практической это удобный компромисс: модель достаточно небольшая для Colab, но поддерживает полноценный fine-tuning. К исходному encoder добавляется классификационная голова с числом выходов, равным количеству предметов.

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = datasets.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample = tokenizer(train_df.iloc[0]["text"], truncation=True, max_length=MAX_LENGTH)
print("Токенов в примере:", len(sample["input_ids"]))
print(tokenizer.convert_ids_to_tokens(sample["input_ids"][:30]))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
model.to(DEVICE)
print("Параметров:", f"{sum(p.numel() for p in model.parameters()):,}")

### 6. Метрики и обучение

Основная метрика — **Macro-F1**, потому что она одинаково учитывает качество по каждому классу. Дополнительно считаем Accuracy, Macro-Precision и Macro-Recall. Лучший checkpoint выбирается по validation Macro-F1.

In [ ]:
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

training_args = TrainingArguments(
    output_dir="rubert_subject_classifier_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_hist = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
eval_hist = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()

if not train_hist.empty:
    axes[0].plot(train_hist["epoch"], train_hist["loss"], marker="o", label="train loss")
if not eval_hist.empty:
    axes[0].plot(eval_hist["epoch"], eval_hist["eval_loss"], marker="o", label="validation loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=.3)

if not eval_hist.empty and "eval_f1_macro" in eval_hist:
    axes[1].plot(eval_hist["epoch"], eval_hist["eval_f1_macro"], marker="o", label="Macro-F1")
if not eval_hist.empty and "eval_accuracy" in eval_hist:
    axes[1].plot(eval_hist["epoch"], eval_hist["eval_accuracy"], marker="o", label="Accuracy")
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1.02)
axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 7. Финальная оценка на отложенном test

Test используется один раз после завершения обучения. Помимо сводных метрик строим classification report и confusion matrix — это позволяет увидеть не только общий результат, но и конкретные пары предметов, которые модель путает.

In [ ]:
pred = trainer.predict(tokenized["test"])
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()

test_metrics = compute_metrics((pred.predictions, y_true))
print("TEST METRICS")
for k, v in test_metrics.items():
    print(f"{k:18s}: {v:.4f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(labels))),
    target_names=labels,
    digits=4,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Confusion matrix — test")
plt.xlabel("Предсказанный класс"); plt.ylabel("Истинный класс")
plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
errors = test_df.reset_index(drop=True).copy()
errors["true"] = [id2label[int(x)] for x in y_true]
errors["pred"] = [id2label[int(x)] for x in y_pred]
errors["confidence"] = probs.max(axis=1)
errors = errors[errors["true"] != errors["pred"]].sort_values("confidence", ascending=False)

print("Ошибок:", len(errors), "из", len(test_df))
display(errors[["title_clean", "true", "pred", "confidence", "text"]].head(15))

### 8. Инференс для ОбрПортал

Следующая функция — уже почти готовое ядро будущего endpoint `POST /api/ml/classify`. На вход приходит текст материала, на выход — основной предмет, confidence и несколько альтернатив.

In [ ]:
def predict_subject(text, top_k=3):
    model.eval()
    encoded = tokenizer(
        text, return_tensors="pt", truncation=True,
        max_length=MAX_LENGTH, padding=True
    ).to(model.device)
    with torch.no_grad():
        logits = model(**encoded).logits[0]
        p = torch.softmax(logits, dim=-1)
    k = min(top_k, len(labels))
    values, indices = torch.topk(p, k=k)
    alternatives = [
        {"category": id2label[int(i)], "confidence": round(float(v), 4)}
        for v, i in zip(values.cpu(), indices.cpu())
    ]
    return {
        "category": alternatives[0]["category"],
        "confidence": alternatives[0]["confidence"],
        "alternatives": alternatives[1:]
    }

examples = [
    "Квадратное уравнение. Рассмотрим дискриминант и формулу корней квадратного уравнения.",
    "Закон Ома. Сила тока, напряжение и электрическое сопротивление участка цепи.",
    "Циклы и условные операторы в языке Python. Алгоритмы и программирование.",
    "Причастный оборот и правила постановки запятых в русском предложении.",
]
for text in examples:
    print(text)
    print(json.dumps(predict_subject(text), ensure_ascii=False, indent=2), "\n")

### 9. Сохранение модели

Сохраняем **весь артефакт**: веса классификатора, конфигурацию и tokenizer. Эту папку затем можно положить в отдельный Python ML-service, не добавляя PyTorch/Transformers в основной backend ОбрПортал.

In [ ]:
EXPORT_DIR = Path("obrportal_subject_classifier")
EXPORT_DIR.mkdir(exist_ok=True)
trainer.save_model(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))

metadata = {
    "base_model": MODEL_NAME,
    "labels": labels,
    "max_length": MAX_LENGTH,
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "seed": SEED,
}
(EXPORT_DIR / "obrportal_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)

archive = shutil.make_archive("obrportal_subject_classifier", "zip", root_dir=str(EXPORT_DIR))
print("Модель сохранена:", EXPORT_DIR.resolve())
print("ZIP:", archive)

### 10. Основа для семантического поиска

Transformer нужен проекту не только как классификатор. Его encoder формирует векторное представление текста. Ниже берём CLS-вектор из **fine-tuned encoder** и демонстрируем поиск похожих учебных материалов по cosine similarity. Это прототип второго этапа ОбрПортал: embeddings + векторный индекс.

In [ ]:
def encode_texts(texts, batch_size=32):
    model.eval()
    vectors = []
    base = model.base_model
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        enc = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=MAX_LENGTH, padding=True
        ).to(model.device)
        with torch.no_grad():
            out = base(**enc)
            vec = out.last_hidden_state[:, 0, :]
            vec = torch.nn.functional.normalize(vec, p=2, dim=1)
        vectors.append(vec.cpu().numpy())
    return np.vstack(vectors)

search_corpus = test_df.reset_index(drop=True).sample(n=min(250, len(test_df)), random_state=SEED).reset_index(drop=True)
corpus_vectors = encode_texts(search_corpus["text"].tolist())

query = "как найти корни уравнения второй степени через дискриминант"
query_vector = encode_texts([query])
scores = cosine_similarity(query_vector, corpus_vectors)[0]
idx = np.argsort(scores)[::-1][:5]

result = search_corpus.loc[idx, ["category", "title_clean", "text"]].copy()
result["similarity"] = scores[idx]
display(result[["category", "title_clean", "similarity"]])

### 11. Итог для практической и проекта

После выполнения всех ячеек зафиксируйте фактические значения Accuracy и Macro-F1, наиболее частые ошибки по confusion matrix и несколько примеров работы `predict_subject()`.

**Практический результат:** обученная модель классификации учебных материалов по предметам.

**Для ОбрПортал:** следующий инженерный шаг — вынести папку `obrportal_subject_classifier` в отдельный ML-service и предоставить REST endpoint классификации. После этого можно добавить embeddings и векторное хранилище для семантического поиска.

In [ ]:
acc = test_metrics["accuracy"]
f1 = test_metrics["f1_macro"]
print("=== АВТОМАТИЧЕСКИЙ ИТОГ ===")
print(f"Количество классов: {len(labels)} — {', '.join(labels)}")
print(f"Train / Validation / Test: {len(train_df)} / {len(val_df)} / {len(test_df)}")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test Macro-F1: {f1:.4f}")
print(f"Ошибок на test: {int((y_true != y_pred).sum())} из {len(y_true)}")
print("\nФормулировка для отчёта:")
print(
    f"В ходе практической работы выполнен fine-tuning русскоязычной трансформерной модели "
    f"{MODEL_NAME} для автоматической классификации учебных материалов по {len(labels)} предметным "
    f"категориям. На независимой тестовой выборке получены Accuracy={acc:.3f} и Macro-F1={f1:.3f}. "
    f"Модель сохранена в переносимом формате и может быть использована как основа отдельного ML-сервиса "
    f"ОбрПортал для автоматического предложения категории материала. Encoder модели также позволяет "
    f"получать embeddings, что создаёт основу для последующего семантического поиска."
)

### 12. Что сдаём преподавателю

Выполненный notebook должен содержать: постановку задачи, описание Transformer, подготовку данных, график распределения классов, fine-tuning, графики обучения, Accuracy/Precision/Recall/Macro-F1, classification report, confusion matrix, анализ ошибок, примеры инференса, сохранение модели и итоговый вывод.

Для проекта сохраняем ZIP с моделью отдельно от Git-репозитория: веса модели лучше хранить как release/artifact/object storage, а в репозитории оставить notebook, код ML-service и инструкцию развёртывания.